In [ ]:
%pip install omnirec

In [12]:
import pandas as pd
import os
"""Pre-processing Check"""
def check_dataset_instances(file_path, dataset_name, separator=','):
    print(f"\n{'='*50}")
    print(f"Überprüfung der Instanzen: {dataset_name}")
    print(f"{'='*50}")

    # 1. Datei laden
    df = pd.read_csv(file_path, sep=separator, header=0)
    print(f"1. Initiale Instanzen (Rohdaten geladen): {len(df)}")

    # 2. Spalten dynamisch zuweisen (Index 0 und 1 für User/Item)
    cols = list(df.columns)
    user_col = cols[ 0 ]
    item_col = cols[ 1 ]

    # Für MovieLens behalten wir vorübergehend die 3. Spalte (Rating)
    if dataset_name == 'MovieLens' and len(cols) >= 3:
        rating_col = cols[ 2 ]
        df = df[[user_col, item_col, rating_col]].copy()
        df.columns = ['user', 'item', 'rating']
    else:
        # Für LastFM nehmen wir nur User und Item
        df = df[[user_col, item_col]].copy()
        df.columns = ['user', 'item']

    # 3. ZUERST: Duplikate entfernen (Canonicalization zu Unique Pairs)
    df = df.drop_duplicates()
    print(f"2. Nach Entfernen der Duplikate:           {len(df)}")

    # 4. DANACH: Rating-Filter (NUR für MovieLens)
    if dataset_name == 'MovieLens':
        df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
        df = df[df['rating'] >= 3]
        print(f"3. Nach '> 3' Rating-Filter:               {len(df)}")
        
        # Rating-Spalte wird für das anschließende Pruning nicht mehr benötigt
        df = df[['user', 'item']].drop_duplicates()
    else:
        print(f"3. Nach '> 3' Rating-Filter:               (Übersprungen für {dataset_name})")

    # 5. Iteratives 5-Core Filtering
    iteration = 1
    while True:
        start_len = len(df)
        
        # User filtern
        user_counts = df['user'].value_counts()
        valid_users = user_counts[user_counts >= 5].index
        df = df[df['user'].isin(valid_users)]
        
        # Items filtern
        item_counts = df['item'].value_counts()
        valid_items = item_counts[item_counts >= 5].index
        df = df[df['item'].isin(valid_items)]
        
        # Abbruchbedingung: Keine Änderungen mehr
        if len(df) == start_len:
            break
        iteration += 1
        
    print(f"4. Nach 5-Core Pruning (Finale Größe):     {len(df)}")
    return len(df)

# ==========================================
# Pfade (Bitte anpassen, falls abweichend)
# ==========================================
"""We converted the u.data of Movielens100k to a .csv and worked with that in this segment. Change the directory to the one, where the .csv is located. path_lfm uses the original user_taggedartists-timestamps.dat file
from HetrecLastFm-dataset. change the directory to the one, where the file is located."""
path_ml = r"D:\uni\PraMaschine\AutoRecLab\sandbox\workspace\working\movielens.csv"
path_lfm = r"D:\uni\PraMaschine\AutoRecLab\sandbox\workspace\working\user_taggedartists-timestamps.dat"

# ==========================================
# Ausführen
# ==========================================
check_dataset_instances(path_ml, 'MovieLens', separator=',')
check_dataset_instances(path_lfm, 'HetrecLastFM', separator='\t')


Überprüfung der Instanzen: MovieLens
1. Initiale Instanzen (Rohdaten geladen): 100000
2. Nach Entfernen der Duplikate:           100000
3. Nach '> 3' Rating-Filter:               82520
4. Nach 5-Core Pruning (Finale Größe):     81697

Überprüfung der Instanzen: HetrecLastFM
1. Initiale Instanzen (Rohdaten geladen): 186479
2. Nach Entfernen der Duplikate:           71064
3. Nach '> 3' Rating-Filter:               (Übersprungen für HetrecLastFM)
4. Nach 5-Core Pruning (Finale Größe):     52551


52551

In [3]:
import os
import re
import json
import copy
import numpy as np
import pandas as pd
from IPython.display import display

# ==============================================================================
# 0. IMPORT LIBRARIES
# ==============================================================================
# This segment imports all required standard libraries for data processing 
# (such as pandas and numpy) as well as the specific modules of the OmniRec 
# framework for dataset loading, preprocessing, and splitting.

# OmniRec Imports
from omnirec import RecSysDataSet
from omnirec.data_loaders.datasets import DataSet
from omnirec.preprocess.pipe import Pipe
from omnirec.preprocess.feedback_conversion import MakeImplicit
from omnirec.preprocess.core_pruning import CorePruning
from omnirec.preprocess.split import UserHoldout
from omnirec.util.util import set_random_state

# ==============================================================================
# 1. DEFINE PATHS AND PARAMETERS
# ==============================================================================
"""
This segment is used to calculate the NDCG@k and Precision@k on the basis of 
the predictions.json files inside the Movielens100k and LastFM checkpoint of one Node. 
This script was used to check all of the Exp B Metrics, and the Metrics of Run 1 & 2 of Exp A.
Run 3 of Exp A was validated by the segment below this one.

It defines the file paths, sets evaluation parameters, and provides helper functions.
"""
CHECKPOINTS_DIR = "/var/mnt/2TB/Dokumente/Projekte/AutoRecLab/AutoRecLab-neu/Artefakte/Gpt5_4/Run 3/sandbox/checkpoint/f6902474abe242f7b4c4f89487e2ba04/working/checkpoints/MovieLens100K-2c020eec/"

# Manueller Overwrite für das Dataset (z. B. "MovieLens100K" oder "LastFM").
# Wenn auf None belassen, wird es automatisch aus CHECKPOINTS_DIR erkannt.
DATASET_NAME = "MovieLens100K"

# Core pruning threshold ( None / 0 if you dont want to have pruning)
CORE_PRUNING_K = 5

# Evaluation Param
K_LIST = [1, 5, 10]

# Manuelle Overrides für den Implicit Threshold (None = Dataset-Standard nutzen: ML=3, LastFM=1)
IMPLICIT_THRESHOLD_OVERRIDE = 4

# Datensatz-Zuordnungen für OmniRec
DATASET_CONFIGS = {
    "MovieLens100K": {
        "enum": DataSet.MovieLens100K,
        "default_threshold": 3,
        "split_folder": "MovieLens100K"
    },
    "LastFM": {
        "enum": DataSet.HetrecLastFM,
        "default_threshold": 1,
        "split_folder": "LastFM"
    }
}

# Helper function for robust type cleaning of item IDs (prevents float/int/str mismatches)
def clean_item_id(val):
    if pd.isna(val):
        return ""
    try:
        return str(int(float(val)))
    except (ValueError, TypeError):
        return str(val).strip()

# ==============================================================================
# 2. METRIC FUNCTION (EXACTLY MATCHING OMNIREC SOURCE CODE)
# ==============================================================================
# This segment contains the logical implementation for calculating the evaluation 
# metrics (NDCG@k and Precision@k). The formulas were replicated exactly from the 
# OmniRec source code to ensure consistency and eliminate metric hallucinations.

def calculate_user_metrics_omnirec_style(recommended_items, relevant_items, k_list):
    metrics = {}
    max_k = max(k_list)
    
    discounted_gain_per_k = np.array(
        [1 / np.log2(i + 1) for i in range(1, max_k + 1)]
    )
    ideal_discounted_gain_per_k = [
        discounted_gain_per_k[: ind + 1].sum()
        for ind in range(len(discounted_gain_per_k))
    ]
    
    pred = [clean_item_id(item) for item in recommended_items[:max_k]]
    rel_set = {clean_item_id(item) for item in relevant_items}
    
    hits = np.isin(pred, list(rel_set))
    user_dcg = np.where(hits, discounted_gain_per_k[:len(hits)], 0)
    
    for k in k_list:
        user_ndcg = user_dcg[:k].sum() / ideal_discounted_gain_per_k[k - 1]
        metrics[f"NDCG@{k}"] = user_ndcg

        top_k = pred[:k]
        prec_hits = [1 if item in rel_set else 0 for item in top_k]
        metrics[f"Precision@{k}"] = sum(prec_hits) / k

    return metrics

# ==============================================================================
# 3. PARSE DIRECTORIES AND CALCULATE METRICS
# ==============================================================================
# In this segment, the checkpoint folders are iteratively searched. For each seed, 
# the dataset is reloaded and preprocessed using the OmniRec pipeline. Subsequently, 
# the predictions are compared with the ground truth (test data), and the metrics 
# are calculated and aggregated per user.

results_list = []
ground_truth_cache = {}

# 1. DATENSATZ BESTIMMEN
active_dataset = DATASET_NAME
if not active_dataset:
    if any(k in CHECKPOINTS_DIR for k in ["LastFM", "HetrecLastFM"]):
        active_dataset = "LastFM"
    else:
        active_dataset = "MovieLens100K"

ds_config = DATASET_CONFIGS.get(active_dataset, DATASET_CONFIGS["MovieLens100K"])
dataset_enum = ds_config["enum"]
split_folder_name = ds_config["split_folder"]
implicit_threshold = (
    IMPLICIT_THRESHOLD_OVERRIDE 
    if IMPLICIT_THRESHOLD_OVERRIDE is not None 
    else ds_config["default_threshold"]
)

# 2. SEED ERMITTELN
path_seed_match = re.search(r"seed_(\d+)", CHECKPOINTS_DIR)
path_seed = int(path_seed_match.group(1)) if path_seed_match else None

if os.path.exists(CHECKPOINTS_DIR):
    subfolders = [
        f for f in os.listdir(CHECKPOINTS_DIR)
        if os.path.isdir(os.path.join(CHECKPOINTS_DIR, f))
    ]

    for folder_name in subfolders:
        if path_seed is not None:
            actual_seed = path_seed
        else:
            folder_parts = folder_name.split("-")
            if folder_parts[-1].isdigit():
                actual_seed = int(folder_parts[-1])
            else:
                continue

        # Algorithmusnamen bestimmen
        raw_algo = folder_name.split("-")[0]
        if "ImplicitMF" in raw_algo:
            algo_name = "ALS"
        elif "ItemKNN" in raw_algo:
            algo_name = "ItemKNN"
        elif "Pop" in raw_algo:
            algo_name = "Pop"
        else:
            algo_name = raw_algo.replace("LensKit.", "").replace("Scorer", "")

        predictions_path = os.path.join(CHECKPOINTS_DIR, folder_name, "predictions.json")
        if not os.path.exists(predictions_path):
            continue

        try:
            df_preds = pd.read_json(predictions_path)
        except Exception:
            continue

        # Ground Truth laden oder generieren
        if actual_seed not in ground_truth_cache:
            saved_split_path = os.path.join(
                "working", "saved_splits", split_folder_name, f"split_seed_{actual_seed}.rsds"
            )

            if os.path.exists(saved_split_path):
                processed_ds = RecSysDataSet.load(saved_split_path)
            else:
                set_random_state(actual_seed)
                
                # DYNAMISCH DIESEN DATENSATZ LADEN (NICHT HARDCODED MOVIELENS!)
                raw_ds = RecSysDataSet.use_dataloader(dataset_enum)
                fresh_ds = copy.deepcopy(raw_ds)

                # DYNAMISCHEN THRESHOLD (1 für LastFM) NUTZEN
                pipe_steps = [MakeImplicit(implicit_threshold)]
                if CORE_PRUNING_K and CORE_PRUNING_K > 0:
                    pipe_steps.append(CorePruning(CORE_PRUNING_K))
                pipe_steps.append(UserHoldout(0.15, 0.15))

                pipe = Pipe(*pipe_steps)
                processed_ds = pipe.process(fresh_ds)

            test_df = processed_ds._data.test
            gt_df = pd.DataFrame({
                "user": test_df["user"].apply(clean_item_id),
                "item": test_df["item"].apply(clean_item_id)
            })
            ground_truth_cache[actual_seed] = (
                gt_df.groupby("user")["item"].apply(set).to_dict()
            )

        user_relevant_map = ground_truth_cache[actual_seed]

        # Predictions auswerten
        if "score" in df_preds.columns:
            df_preds = df_preds.sort_values(by=["user", "score"], ascending=[True, False])
        elif "rank" in df_preds.columns:
            df_preds = df_preds.sort_values(by=["user", "rank"], ascending=[True, True])

        df_preds["user_clean"] = df_preds["user"].apply(clean_item_id)
        df_preds["item_clean"] = df_preds["item"].apply(clean_item_id)

        user_preds_map = df_preds.groupby("user_clean")["item_clean"].apply(list).to_dict()

        all_user_metrics = []
        for user_id, rec_items in user_preds_map.items():
            rel_items = user_relevant_map.get(user_id, set())
            if not rel_items:
                continue

            u_metrics = calculate_user_metrics_omnirec_style(rec_items, rel_items, K_LIST)
            all_user_metrics.append(u_metrics)

        if all_user_metrics:
            df_user_res = pd.DataFrame(all_user_metrics)
            mean_metrics = df_user_res.mean().to_dict()

            row_data = {"Seed": actual_seed, "Algorithm": algo_name}
            row_data.update(mean_metrics)
            results_list.append(row_data)
# ==============================================================================
# 4. SORT AND OUTPUT RESULTS
# ==============================================================================
# This final segment aggregates the collected metrics from the loop, cleanly sorts 
# the results by seed as well as algorithm, and formats the output as a final table.

if results_list:
    df_final = pd.DataFrame(results_list)
    df_final = df_final.sort_values(by=['Seed', 'Algorithm']).reset_index(drop=True)
    
    metric_cols = [col for col in df_final.columns if col not in ['Seed', 'Algorithm']]
    metric_cols = sorted(metric_cols, key=lambda x: ("NDCG" not in x, int(x.split('@')[-1])))
    
    df_final = df_final[['Seed', 'Algorithm'] + metric_cols]
    
    print("\n" + "=" * 80)
    print("  AGGREGATED METRICS (OMNIREC-STYLE NDCG FORMULA) SORTED BY SEED  ")
    print("=" * 80)
    display(df_final.round(6))
else:
    print("No evaluable folders or predictions found.")

[2026/08/02 21:02:07] INFO     Canonicalized data set already exists, skipping download and  ]8;id=13806285;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/recsys_data_set.py\recsys_data_set.py]8;;\:]8;id=13806286;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/recsys_data_set.py#205\205]8;;\
                               canonicalization.                                                                   

                      INFO     Making data set implicit with threshold 4.                 ]8;id=13806291;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/feedback_conversion.py\feedback_conversion.py]8;;\:]8;id=13806292;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/feedback_conversion.py#21\21]8;;\

                      INFO     Minimum rating: 1                                          ]8;id=13806297;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/feedback_conversion.py\feedback_conversion.py]8;;\:]8;id=13806298;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/feedback_conversion.py#24\24]8;;\

                      INFO     Maximum rating: 5                                          ]8;id=13806303;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/feedback_conversion.py\feedback_conversion.py]8;;\:]8;id=13806304;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/feedback_conversion.py#25\25]8;;\

                      INFO     Number of interactions before: 100000                      ]8;id=13806309;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/feedback_conversion.py\feedback_conversion.py]8;;\:]8;id=13806310;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/feedback_conversion.py#26\26]8;;\

...

                      INFO     Pruning data set to 5-core.                                       ]8;id=13806537;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/core_pruning.py\core_pruning.py]8;;\:]8;id=13806538;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/core_pruning.py#19\19]8;;\

                      INFO     Number of interactions before: 55375                              ]8;id=13806543;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/core_pruning.py\core_pruning.py]8;;\:]8;id=13806544;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/core_pruning.py#20\20]8;;\

                      INFO     Number of interactions after: 54413                               ]8;id=13806549;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/core_pruning.py\core_pruning.py]8;;\:]8;id=13806550;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/core_pruning.py#63\63]8;;\


  AGGREGATED METRICS (OMNIREC-STYLE NDCG FORMULA) SORTED BY SEED  


,Seed,Algorithm,NDCG@1,NDCG@5,NDCG@10,Precision@1,Precision@5,Precision@10
0,11,ALS,0.061043,0.043428,0.036890,0.061043,0.038180,0.031410
1,11,ItemKNN,0.067703,0.051359,0.044655,0.067703,0.047059,0.039623
2,11,Pop,0.048835,0.037673,0.032065,0.048835,0.034184,0.027858
3,23,ALS,0.041020,0.044311,0.038243,0.041020,0.044568,0.035698
4,23,ItemKNN,0.078714,0.056471,0.046087,0.078714,0.050776,0.038803
5,23,Pop,0.047672,0.034084,0.030399,0.047672,0.030377,0.026940
6,37,ALS,0.044643,0.033096,0.030763,0.044643,0.030357,0.028237
7,37,ItemKNN,0.053571,0.046689,0.041056,0.053571,0.044866,0.037500
8,37,Pop,0.056856,0.041921,0.034756,0.056856,0.037904,0.029654
9,49,ALS,0.058242,0.043272,0.037387,0.058242,0.039560,0.032857


In [5]:
import os
import re
import pandas as pd
import numpy as np
from IPython.display import display

# OmniRec Importe
from omnirec import RecSysDataSet
from omnirec.data_loaders.datasets import DataSet
from omnirec.preprocess.pipe import Pipe
from omnirec.preprocess.feedback_conversion import MakeImplicit
from omnirec.preprocess.core_pruning import CorePruning
# NEU für Run 3: RandomHoldout und RatingFilter
from omnirec.preprocess.split import RandomHoldout
from omnirec.preprocess.filter import RatingFilter
from omnirec.util.util import set_random_state

# ==============================================================================
# DOKUMENTATION: ÜBERPRÜFUNG DER METRIKEN (RUN 3)
# ==============================================================================
# Dieses Skript dient der unabhängigen Validierung der Ranking-Metriken für Run 3.
# Es liest die von AutoRecLab erzeugten Modell-Vorhersagen ('predictions.json') ein 
# und vergleicht sie mit den Ground-Truth-Daten. 
# WICHTIG: Im Gegensatz zu Run 1 verwendet Run 3 einen globalen 'RandomHoldout'-
# Splitter sowie einen initialen 'RatingFilter(lower=4)' vor der Umwandlung 
# in implizites Feedback. Diese exakte Pipeline wird hier nachgebaut, um einen 
# Ground-Truth-Mismatch auszuschließen.
# ==============================================================================

# 1. PFADE UND PARAMETER DEFINIEREN
# Pfad zum MovieLens-Checkpoint von Run 3
CHECKPOINTS_DIR = "/var/mnt/2TB/Dokumente/Projekte/AutoRecLab/AutoRecLab-neu/Artefakte/Gpt5_4/Run 3/sandbox/checkpoint/f6902474abe242f7b4c4f89487e2ba04/working/checkpoints/MovieLens100K-2c020eec/"

K_LIST = [1, 5, 10]
SPLIT_VAL = 0.15          
SPLIT_TEST = 0.15         

def clean_item_id(val):
    """Hilfsfunktion zur robusten Typbereinigung von Item-IDs."""
    if pd.isna(val):
        return ""
    try:
        return str(int(float(val)))
    except (ValueError, TypeError):
        return str(val).strip()

# ==============================================================================
# 2. METRIK-FUNKTION (EXAKT NACH OMNIREC-QUELLCODE)
# ==============================================================================
def calculate_user_metrics_omnirec_style(recommended_items, relevant_items, k_list):
    metrics = {}
    max_k = max(k_list)
    
    discounted_gain_per_k = np.array(
        [1 / np.log2(i + 1) for i in range(1, max_k + 1)]
    )
    ideal_discounted_gain_per_k = [
        discounted_gain_per_k[: ind + 1].sum()
        for ind in range(len(discounted_gain_per_k))
    ]
    
    pred = [clean_item_id(item) for item in recommended_items[:max_k]]
    rel_set = {clean_item_id(item) for item in relevant_items}
    
    hits = np.isin(pred, list(rel_set))
    user_dcg = np.where(hits, discounted_gain_per_k[:len(hits)], 0)
    
    for k in k_list:
        user_ndcg = user_dcg[:k].sum() / ideal_discounted_gain_per_k[k - 1]
        metrics[f"NDCG@{k}"] = user_ndcg

        top_k = pred[:k]
        prec_hits = [1 if item in rel_set else 0 for item in top_k]
        metrics[f"Precision@{k}"] = sum(prec_hits) / k

    return metrics

# ==============================================================================
# 3. ORDNER DURCHSUCHEN, GROUND-TRUTH REKONSTRUIEREN UND METRIKEN BERECHNEN
# ==============================================================================
results_list = []
folder_pattern = re.compile(r"^(.+?)-[^-]+-(.+)$")
ground_truth_cache = {}

if os.path.exists(CHECKPOINTS_DIR):
    subfolders = [f for f in os.listdir(CHECKPOINTS_DIR) if os.path.isdir(os.path.join(CHECKPOINTS_DIR, f))]
    
    for folder_name in subfolders:
        match = folder_pattern.match(folder_name)
        if not match:
            continue
        
        raw_algo = match.group(1)
        seed = int(match.group(2))
        algo_name = raw_algo.replace("LensKit.", "").replace("Scorer", "")
        
        predictions_path = os.path.join(CHECKPOINTS_DIR, folder_name, "predictions.json")
        if not os.path.exists(predictions_path):
            continue
            
        try:
            df_preds = pd.read_json(predictions_path)
        except Exception as e:
            continue

        # Pipeline exakt nach Run 3 nachbauen
        if seed not in ground_truth_cache:
            set_random_state(seed)
            raw_ds = RecSysDataSet.use_dataloader(DataSet.MovieLens100K)
            
            # Die Reihenfolge der Schritte ist entscheidend für den korrekten Split!
            pipe = Pipe(
                RatingFilter(lower=4), 
                MakeImplicit(3), 
                CorePruning(5), 
                RandomHoldout(validation_size=SPLIT_VAL, test_size=SPLIT_TEST)
            )
            
            processed_ds = pipe.process(raw_ds)
            test_df = processed_ds._data.test
            ground_truth_cache[seed] = test_df.groupby('user')['item'].apply(set).to_dict()
            
        user_relevant_map = ground_truth_cache[seed]
            
        # Predictions ordnen
        if 'score' in df_preds.columns:
            df_preds = df_preds.sort_values(by=['user', 'score'], ascending=[True, False])
        elif 'rank' in df_preds.columns:
            df_preds = df_preds.sort_values(by=['user', 'rank'], ascending=[True, True])
            
        user_preds_map = df_preds.groupby('user')['item'].apply(list).to_dict()
        
        # Metriken auf Nutzer-Ebene berechnen
        all_user_metrics = []
        for user_id, rec_items in user_preds_map.items():
            rel_items = user_relevant_map.get(user_id, set())
            if not rel_items: 
                continue
            
            u_metrics = calculate_user_metrics_omnirec_style(rec_items, rel_items, K_LIST)
            all_user_metrics.append(u_metrics)
            
        # Ergebnisse aggregieren
        if all_user_metrics:
            df_user_res = pd.DataFrame(all_user_metrics)
            mean_metrics = df_user_res.mean().to_dict()
            
            row_data = {
                'Seed': seed,
                'Algorithm': algo_name
            }
            row_data.update(mean_metrics)
            results_list.append(row_data)

# ==============================================================================
# 4. ERGEBNISSE SORTIEREN UND AUSGEBEN
# ==============================================================================
if results_list:
    df_final = pd.DataFrame(results_list)
    df_final = df_final.sort_values(by=['Seed', 'Algorithm']).reset_index(drop=True)
    
    metric_cols = [col for col in df_final.columns if col not in ['Seed', 'Algorithm']]
    metric_cols = sorted(metric_cols, key=lambda x: ("NDCG" not in x, int(x.split('@')[-1])))
    
    df_final = df_final[['Seed', 'Algorithm'] + metric_cols]
    
    print("\n" + "=" * 80)
    print("  RUN 3 - AGGREGATED METRICS (OMNIREC-STYLE NDCG FORMULA) SORTED BY SEED  ")
    print("=" * 80)
    display(df_final.round(6))
else:
    print("Keine auswertbaren Ordner oder Predictions gefunden.")

[2026/08/02 21:10:37] INFO     Canonicalized data set already exists, skipping download and  ]8;id=13806825;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/recsys_data_set.py\recsys_data_set.py]8;;\:]8;id=13806826;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/recsys_data_set.py#205\205]8;;\
                               canonicalization.                                                                   

                      INFO     Making data set implicit with threshold 3.                 ]8;id=13806831;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/feedback_conversion.py\feedback_conversion.py]8;;\:]8;id=13806832;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/feedback_conversion.py#21\21]8;;\

                      INFO     Minimum rating: 4                                          ]8;id=13806837;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/feedback_conversion.py\feedback_conversion.py]8;;\:]8;id=13806838;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/feedback_conversion.py#24\24]8;;\

                      INFO     Maximum rating: 5                                          ]8;id=13806843;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/feedback_conversion.py\feedback_conversion.py]8;;\:]8;id=13806844;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/feedback_conversion.py#25\25]8;;\

                      INFO     Number of interactions before: 55375                       ]8;id=13806849;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/feedback_conversion.py\feedback_conversion.py]8;;\:]8;id=13806850;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/feedback_conversion.py#26\26]8;;\

...

                      INFO     Pruning data set to 5-core.                                       ]8;id=13807077;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/core_pruning.py\core_pruning.py]8;;\:]8;id=13807078;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/core_pruning.py#19\19]8;;\

                      INFO     Number of interactions before: 55375                              ]8;id=13807083;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/core_pruning.py\core_pruning.py]8;;\:]8;id=13807084;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/core_pruning.py#20\20]8;;\

                      INFO     Number of interactions after: 54413                               ]8;id=13807089;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/core_pruning.py\core_pruning.py]8;;\:]8;id=13807090;file:///home/Mike-bazz/.local/lib/python3.14/site-packages/omnirec/preprocess/core_pruning.py#63\63]8;;\


  RUN 3 - AGGREGATED METRICS (OMNIREC-STYLE NDCG FORMULA) SORTED BY SEED  


,Seed,Algorithm,NDCG@1,NDCG@5,NDCG@10,Precision@1,Precision@5,Precision@10
0,11,ImplicitMF,0.155383,0.127880,0.112359,0.155383,0.119423,0.101221
1,11,ItemKNN,0.224195,0.170893,0.144522,0.224195,0.156715,0.125638
2,11,Pop,0.138735,0.108946,0.094205,0.138735,0.100333,0.083130
3,23,ImplicitMF,0.155211,0.125144,0.109413,0.155211,0.119734,0.099889
4,23,ItemKNN,0.210643,0.157124,0.132120,0.210643,0.143016,0.113969
5,23,Pop,0.140798,0.105262,0.092774,0.140798,0.095787,0.082262
6,37,ImplicitMF,0.137277,0.114261,0.101604,0.137277,0.107812,0.092969
7,37,ItemKNN,0.190848,0.157158,0.135395,0.190848,0.149554,0.122210
8,37,Pop,0.138239,0.110977,0.097271,0.138239,0.103010,0.087402
9,49,ImplicitMF,0.134066,0.122934,0.106311,0.134066,0.118901,0.096813
